# JAC279 방식 IPM 모터 열해석 — 3D FEM + 열등가회로 (PyMAPDL)

**대상**: AEDT `Electric_Motor_Mechanical_AEDT_3D_part1 :: IcepakFEADesign1`
(8극/48슬롯 IPM, 실측 형상) — 사용자가 지칭한 **"3번째(열) 디자인"**.

JMAG **JAC279** *"Thermal Analysis Accounting for Cooling of the IPM Motor"* 의
**3D 열 FEM + 열등가회로 하이브리드** 개념을 Ansys MAPDL 로 재현한다.

| JMAG | MAPDL |
|------|-------|
| 3D 솔리드 열 FEM | `SOLID87` (10절점 사면체 열요소) |
| Heat Transfer Boundary (Referred by Circuit Component) | `SURF152` + extra node (`KEYOPT(5)=1`) |
| Thermal Resistor (R) | `COMBIN14`, `KEYOPT(2)=8`, 실상수=열컨덕턴스 [W/degC] |
| Heat Capacity (C) | `MASS71`, `KEYOPT(3)=1`, 실상수=열용량 [J/degC] |
| Fixed Temperature (WJ/ATF/외기) | `D, TEMP` |

**해석 전략** — 능동부(코어·자석·코일·샤프트)는 **3D FEM**, 하우징·냉각계는 **열회로(lumped)**.
원 형상은 **1/8 주기섹터**(8극 중 1극)이므로, MAPDL 에서 **45°×8 로 패턴하여 360°** 로 만든 뒤
JAC279식 비대칭 냉각(상부 90° 워터재킷 / 하부 90° ATF)을 적용한다.

**두 가지 실행 경로** (공통: 4~8장):
- **CAD 경로 (실형상, 본 타깃)** — 3장 스킵 → **3B-1, 3B-2** 실행 → 4장~
- **링 경로 (검증용 fallback)** — 3B 스킵 → **3장** 실행 → 4장~. 처음엔 링으로 전체 흐름 확인 권장.

```
conda activate pymotorenv_310   # (또는 PyMotorEnv_310)
jupyter lab jac279_fem_network_pymapdl.ipynb
```
> CAD 경로는 대상 AEDT 프로젝트가 **열려 있어야** 한다(이름으로 attach). `~SATIN` 임포트에는
> **ACIS Geometry Interface** 라이선스가 필요하다.

In [ ]:
import os
import math
import csv

from ansys.mapdl.core import launch_mapdl

## 0. 파라미터 — 실측 기하(AEDT 설계변수) + JAC279 회로 정수

기하/재료는 AEDT `Electric_Motor_Mechanical_AEDT_3D_part1` 설계변수에서 추출.
손실값은 **JAC279 예시(placeholder)** 이니 이 모터의 Maxwell 손실 리포트로 교체할 것.

In [ ]:
# ── 실측 기하 (AEDT 설계변수, 단위 m) ─────────────────────────────────────
R_SHAFT   = 0.04445 / 2      # DiaShaft   44.45 mm  -> 0.022225
R_ROT_OUT = 0.130   / 2      # RotorDia  130 mm    -> 0.065 (로터 적층 외경)
GAP       = 0.001            # Airgap      1 mm
R_STA_IN  = 0.132   / 2      # DiaGap    132 mm    -> 0.066 (스테이터 보어)
R_STA_OUT = 0.198   / 2      # DiaOuter  198 mm    -> 0.099 (스테이터 외경/요크)
STACK     = 0.160            # Stator_Lam_Length (열해석 축길이 기준)
ROT_STACK = 0.150            # Rotor_Lam_Length
NPOLE     = 8                # 8 극
NSLOT     = 48               # 48 슬롯
N_SECTOR  = 8                # 1/8 섹터 -> 45°×8 패턴
SEC_ANG   = 360.0 / N_SECTOR # 45 deg

# 코일(슬롯) 반경 밴드 — 슬롯 깊이 미상, 링근사/평가점용 추정치
R_ROT_IN   = 0.050           # 로터코어 내측(자석층 내경, 링근사)
R_MAG_OUT  = 0.060           # 자석층 외경(링근사)
R_COIL_IN  = 0.067           # 슬롯 코일 내경(스테이터 보어 근처)
R_COIL_OUT = 0.083           # 슬롯 코일 외경(요크 아래)
COILEND_H  = 0.0475          # 엔드와인딩 축방향 돌출 근사((Model_Length-STACK)/2)

# ── 재료 (AEDT) ──────────────────────────────────────────────────────────
K_CORE,  CP_CORE,  RHO_CORE  = 23.0, 460.0, 7650.0    # M350-50A (전기강판)
K_MAG,   CP_MAG,   RHO_MAG   =  9.0, 460.0, 7500.0    # N30UH NdFeB (관통방향 ~9 W/mK)
K_COIL,  CP_COIL,  RHO_COIL_SOLID = 380.0, 380.0, 8960.0   # Copper
FILL     = 0.45                                       # 슬롯 점적률
RHO_COIL = RHO_COIL_SOLID * FILL
K_SHAFT, CP_SHAFT, RHO_SHAFT = 52.0, 460.0, 7870.0    # Steel_1008 (WS01_2 기준)

# 재료번호:  1=스테이터코어  2=자석  3=코일  4=샤프트  5=로터코어(1과 동일물성, 손실분리용)
MAT_CORE_S, MAT_MAG, MAT_COIL, MAT_SHAFT, MAT_CORE_R = 1, 2, 3, 4, 5

# ── 손실 [W] : ⚠ JAC279 예시값(placeholder). 이 모터 Maxwell 손실로 교체 ──
LOSS_COPPER      = 1004.0
LOSS_MAGNET      = 71.0
LOSS_IRON_ROTOR  = 65.0
LOSS_IRON_STATOR = 880.0

# ── 열등가회로 정수 (JAC279 Table A-1/A-2, 하우징 lumped) ─────────────────
T_INIT   = 70.0             # 초기/외기/WJ/ATF 온도 [degC]
HTC_AIR  = 10.0             # 내부공기 열전달계수 [W/m2K]
HTC_ATF  = 300.0           # ATF 열전달계수
HTC_BIG  = 10000.0         # 경계자체 저항 제거용 큰 값
TC_AIR   = 0.03            # 공기 열전도율
H_CONTACT = TC_AIR / 10e-6 # 스테이터-하우징 Good fit 0.01mm -> 3000 W/m2K
C_SHAFT  = 3771.231        # 샤프트 열용량 [J/K]
R_SHAFT_TH = 5.3759        # 샤프트 열저항 [K/W]
C_HOUSING = 12759.55       # 하우징 총 열용량 [J/K]
R_HOUS_AMB = 0.3430063     # 하우징-외기 열저항
R_HOUS_AXIAL = 2.294125    # 하우징 축방향 열저항

# 워터재킷 (JAC279 Table 2-4): Dittus-Boelter
WJ_V, WJ_W, WJ_H, WJ_NTUBE, WJ_R = 4.0, 0.010, 0.007, 3, 0.111
RHO_W, MU_W, K_W, CP_W = 978.0, 4.04e-4, 0.662, 4190.0
DH = 2 * WJ_W * WJ_H / (WJ_W + WJ_H)
RE = RHO_W * WJ_V * DH / MU_W
PR = MU_W * CP_W / K_W
NU = 0.023 * RE**0.8 * PR**0.4
HTC_WJ = NU * K_W / DH
A_WJ = WJ_NTUBE * 2 * (WJ_W + WJ_H) * (math.pi / 2 * WJ_R)
G_WJ = HTC_WJ * A_WJ

# ── 해석 조건 ────────────────────────────────────────────────────────────
T_END = 900.0              # 종료시각 [s]
DT    = 45.0               # 시간증분
ESIZE = 0.008              # 요소크기 [m] (데모용; 정밀시 축소)
TOL   = 1e-5               # 선택 허용오차

# 냉각 섹터 각도 (CSYS,1 의 θ 는 -180..180 deg)
TH_WJ   = [(45.0, 135.0)]                                    # 워터재킷(상부 90°)
TH_ATF  = [(-135.0, -45.0)]                                  # ATF 풀(하부 90°)
TH_REST = [(-45.0, 45.0), (135.0, 180.0), (-180.0, -135.0)] # 나머지 180°

print(f"HTC_WJ = {HTC_WJ:.1f} W/m2K,  G_WJ = {G_WJ:.2f} W/K,  Re = {RE:.0f}")
print(f"Stator OD/ID = {R_STA_OUT*2000:.0f}/{R_STA_IN*2000:.0f} mm, "
      f"Rotor OD = {R_ROT_OUT*2000:.0f} mm, Shaft OD = {R_SHAFT*2000:.1f} mm")

## 1. MAPDL 기동
라이선스 서버가 잡히지 않아 *VERIFICATION RUN* 으로 뜨면 `~SATIN`/대형해석이 막힌다.
필요 시 `os.environ["ANSYSLMD_LICENSE_FILE"]="<port>@<license-server>"` 를 먼저 설정.

In [ ]:
# 라이선스 서버가 환경변수에 없으면 아래 주석을 해제해 지정
# os.environ["ANSYSLMD_LICENSE_FILE"] = "<port>@<license-server>"   # 예: 1055@10.x.x.x

mapdl = launch_mapdl(run_location=os.path.join(os.getcwd(), "mapdl_work"),
                     override=True, loglevel="WARNING")
print(mapdl)
mapdl.clear()
mapdl.prep7()
mapdl.units("SI")

## 2. 요소타입 / 재료 (mat 1~5)

In [ ]:
mapdl.et(1, "SOLID87")          # 3D 10절점 사면체 열전도
mapdl.et(2, "SURF152")          # 표면효과(대류)
mapdl.keyopt(2, 5, 1)           # extra node <- 회로노드 온도 참조
mapdl.keyopt(2, 8, 2)           # ⚠ 필수: 대류 표면하중 '포함' (기본값 0 은 CONV 무시!
                                #   -> 회로 미결합/단열 과열/GAP 노드 부동의 원인)
mapdl.et(3, "COMBIN14")         # 열저항(스프링->열컨덕턴스)
mapdl.keyopt(3, 2, 8)           # TEMP DOF
mapdl.et(4, "MASS71")           # 열용량
mapdl.keyopt(4, 3, 1)           # 실상수=열용량 직접입력

def set_mat(n, k, c, rho):
    mapdl.mp("KXX", n, k); mapdl.mp("C", n, c); mapdl.mp("DENS", n, rho)

set_mat(MAT_CORE_S, K_CORE,  CP_CORE,  RHO_CORE)    # 1 스테이터코어
set_mat(MAT_MAG,    K_MAG,   CP_MAG,   RHO_MAG)     # 2 자석
set_mat(MAT_COIL,   K_COIL,  CP_COIL,  RHO_COIL)    # 3 코일
set_mat(MAT_SHAFT,  K_SHAFT, CP_SHAFT, RHO_SHAFT)   # 4 샤프트
set_mat(MAT_CORE_R, K_CORE,  CP_CORE,  RHO_CORE)    # 5 로터코어(동일물성)
print("materials 1..5 defined")

## 3. (링 경로) 동심 링 360° — 검증용 fallback
실측 반경 기반 동심 링. **CAD 경로(3B)를 쓸 경우 이 셀과 다음 메시확인 셀을 건너뛴다.**

In [ ]:
Z1, Z2 = -STACK / 2, STACK / 2
rings = [
    (R_SHAFT,   R_ROT_IN,   Z1, Z2, MAT_CORE_R),   # 로터코어 내측
    (R_ROT_IN,  R_MAG_OUT,  Z1, Z2, MAT_MAG),      # 자석(균질화 링)
    (R_MAG_OUT, R_ROT_OUT,  Z1, Z2, MAT_CORE_R),   # 로터코어 브리지
    (R_STA_IN,  R_COIL_IN,  Z1, Z2, MAT_CORE_S),   # 스테이터 치선단
    (R_COIL_IN, R_COIL_OUT, Z1, Z2, MAT_COIL),     # 코일(슬롯부)
    (R_COIL_OUT,R_STA_OUT,  Z1, Z2, MAT_CORE_S),   # 스테이터 요크
    (R_COIL_IN, R_COIL_OUT, Z2, Z2 + COILEND_H, MAT_COIL),  # 코일엔드 상
    (R_COIL_IN, R_COIL_OUT, Z1 - COILEND_H, Z1, MAT_COIL),  # 코일엔드 하
]
for (r1, r2, z1, z2, _mat) in rings:
    for th in (45, 135, 225, 315):
        mapdl.cylind(r1, r2, z1, z2, th, th + 90)
mapdl.vglue("ALL")
mapdl.numcmp("VOLU")

# 재료 할당: 각 볼륨의 키포인트 반경(rmin,rmax)으로 링밴드 판별 (centroid 기반 VSEL 오분류 회피)
mapdl.csys(0)
vmax = int(mapdl.get_value("VOLU", 0, "NUM", "MAX"))
vol_band = {}
for vid in range(1, vmax + 1):
    mapdl.vsel("S", "VOLU", "", vid); mapdl.aslv("S"); mapdl.lsla("S"); mapdl.ksll("S")
    radii = []
    k = int(mapdl.get_value("KP", 0, "NUM", "MIN"))
    while k > 0:
        x = mapdl.get_value("KP", k, "LOC", "X"); y = mapdl.get_value("KP", k, "LOC", "Y")
        radii.append(math.hypot(x, y))
        k = int(mapdl.get_value("KP", k, "NXTH"))
    vol_band[vid] = (min(radii), max(radii))
mapdl.allsel()

def sel_band_volumes(r1, r2):
    mapdl.vsel("NONE")
    for vid, (a, b) in vol_band.items():
        if abs(a - r1) < 1e-4 and abs(b - r2) < 1e-4:
            mapdl.vsel("A", "VOLU", "", vid)

for (r1, r2, _z1, _z2, mat) in rings:
    sel_band_volumes(r1, r2)
    mapdl.vatt(mat, 0, 1)

mapdl.allsel()
mapdl.esize(ESIZE); mapdl.mshape(1, "3D"); mapdl.mshkey(0)
mapdl.vmesh("ALL")
print(f"[ring] meshed: {mapdl.mesh.n_node} nodes, {mapdl.mesh.n_elem} elems")

### (링) 메시 확인

In [ ]:
mapdl.eplot(vtk=True, show_edges=False)

## 3B. (CAD 경로 — 실형상, 본 타깃) AEDT IcepakFEA 형상 → MAPDL

현재 열려있는 AEDT `Electric_Motor_Mechanical_AEDT_3D_part1 :: IcepakFEADesign1`(1/8 섹터)의
솔리드를 재료군별로 `.sat` export → MAPDL 로 임포트 → **45°×8 패턴(360°)** → glue → 메시.

**3장(링) 대신 아래 3B-1, 3B-2 를 실행**한다.

- 실측: 46 솔리드 = 스테이터적층1 + 로터적층1 + 자석36 + 코일5 + 샤프트1
  (`Rotating_Band_out`, `Whole_Region` = 공기영역, 임포트 제외)
- `~SATIN` 임포트에는 **ACIS Geometry Interface** 라이선스 필요.

In [ ]:
# ── 3B-1. PyAEDT(0.24) 로 열린 IcepakFEA 디자인에 attach -> 재료군별 .sat export ──
from ansys.aedt.core import Mechanical

AEDT_VER = "2026.1"                                     # 실행 중 AEDT 버전(v261)
PROJ     = "Electric_Motor_Mechanical_AEDT_3D_part1"
DESIGN   = "IcepakFEADesign1"                            # = "3번째(열) 디자인"

mech = Mechanical(version=AEDT_VER, new_desktop=False, close_on_exit=False,
                  project=PROJ, design=DESIGN)
print("attached:", mech.design_name, "| type:", mech.design_type)

solids = list(mech.oeditor.GetObjectsInGroup("Solids"))
def pick(pred): return [n for n in solids if pred(n)]
groups = {
    "stator_core": pick(lambda n: "Stator_Lamination" in n),
    "rotor_core":  pick(lambda n: "Rotor_Lamination" in n),
    "magnet":      pick(lambda n: n.startswith("Magnet")),
    "coil":        pick(lambda n: n[:3] in ("Ph1", "Ph2", "Ph3")),
    "shaft":       pick(lambda n: "Shaft" in n),
}
covered = set().union(*groups.values())
print("ungrouped(제외):", [n for n in solids if n not in covered])   # 공기영역이어야 함

EXP = os.path.abspath("aedt_export")
os.makedirs(EXP, exist_ok=True)
for g, objs in groups.items():
    if not objs:
        print(f"[skip] {g}: 오브젝트 없음"); continue
    mech.modeler.export_3d_model(file_name=g, file_path=EXP, file_format=".sat",
                                 assignment_to_export=objs)
    fp = os.path.join(EXP, g + ".sat")
    print(f"[ok] {g}: {len(objs)} objs -> {g}.sat ({os.path.getsize(fp)} B)")

mech.release_desktop(close_projects=False, close_on_exit=False)   # AEDT 는 열어둠
print("released (AEDT 세션 유지)")

In [ ]:
# ── 3B-2. MAPDL 임포트 -> 45°×8 패턴(360°) -> glue -> 재료할당 -> 메시 ──
EXP = os.path.abspath("aedt_export")

# (group_file, 재료번호)  ※ 재료물성은 2장에서 정의됨
GROUP_MAT = [
    ("stator_core", MAT_CORE_S),
    ("rotor_core",  MAT_CORE_R),
    ("magnet",      MAT_MAG),
    ("coil",        MAT_COIL),
    ("shaft",       MAT_SHAFT),
]

def vmaxd():
    return int(mapdl.get_value("VOLU", 0, "NUM", "MAXD"))

mapdl.csys(1)                      # 원통좌표: VGEN 의 DY = θ 증분(deg)
for gf, mat in GROUP_MAT:
    fp = os.path.join(EXP, gf + ".sat")
    if not os.path.exists(fp):
        print(f"[skip] {gf}: {fp} 없음"); continue
    v0 = vmaxd()
    mapdl.run(f"~SATIN,'{gf}','sat','{EXP}',SOLIDS,0,0")   # ACIS 임포트
    v1 = vmaxd()
    if v1 <= v0:
        raise RuntimeError(f"{gf}: ~SATIN 임포트 실패(볼륨 0). ACIS Geometry Interface "
                           f"라이선스/번역기 확인, 또는 IGES(igesin) 로 대체.")
    # 45°×8 회전복제 (원본 포함 8개) -> 360°
    mapdl.vsel("S", "VOLU", "", v0 + 1, v1)
    if N_SECTOR > 1:
        mapdl.vgen(N_SECTOR, "ALL", "", "", "", SEC_ANG, "", "", 0)
    v2 = vmaxd()
    # 재료 즉시 할당 (glue 전에 baked-in)
    mapdl.vsel("S", "VOLU", "", v0 + 1, v2)
    mapdl.vatt(mat, 0, 1)
    print(f"[{gf}] 섹터 {v1 - v0}개 -> 패턴 후 {v2 - v0}개 볼륨, mat={mat}")
mapdl.csys(0)
mapdl.allsel()

# 부품간 계면 conformal 화 (VGLUE 는 자식볼륨에 MAT 속성 유지)
mapdl.vglue("ALL")
mapdl.allsel()

# 메시
mapdl.esize(ESIZE); mapdl.mshape(1, "3D"); mapdl.mshkey(0)
mapdl.vmesh("ALL")
print(f"[CAD] meshed: {mapdl.mesh.n_node} nodes, {mapdl.mesh.n_elem} elems")

# 재료별 볼륨/요소 수 확인
for mat in (MAT_CORE_S, MAT_CORE_R, MAT_MAG, MAT_COIL, MAT_SHAFT):
    mapdl.esel("S", "MAT", "", mat)
    print(f"  mat {mat}: {mapdl.mesh.n_elem} elems")
mapdl.allsel()

## 4. 열등가회로 노드 (JAC279 Fig.4-2 축약)
하우징·샤프트·공극·냉각 고정온도를 회로(COMBIN14/MASS71/D)로 구성. 링/CAD 공통.

In [ ]:
nmax = int(mapdl.get_value("NODE", 0, "NUM", "MAXD"))
def net_node(i):
    n = nmax + i
    mapdl.csys(0)
    mapdl.n(n, 0.5 + 0.02 * i, 0, 0)   # 모델 밖 식별용 위치
    return n

# 회로 노드 (이름 -> MAPDL 절점번호). 10장 시각화에서 재사용.
NET_NODES = {nm: net_node(i + 1) for i, nm in enumerate(
    ["WJ", "ATF", "AMB", "AIR", "SHF", "H_WJ", "H_ATF", "H_RST", "GAP_S", "GAP_R"])}
N_WJ, N_ATF, N_AMB = NET_NODES["WJ"], NET_NODES["ATF"], NET_NODES["AMB"]
N_AIR, N_SHF = NET_NODES["AIR"], NET_NODES["SHF"]
N_H_WJ, N_H_ATF, N_H_RST = NET_NODES["H_WJ"], NET_NODES["H_ATF"], NET_NODES["H_RST"]
N_GAP_S, N_GAP_R = NET_NODES["GAP_S"], NET_NODES["GAP_R"]

_rid = [100]
def add_C(node, c):
    _rid[0] += 1
    mapdl.type(4); mapdl.real(_rid[0]); mapdl.r(_rid[0], c); mapdl.e(node)

# 열용량 (이름 -> J/K). 10장 시각화에서 재사용.
NET_CAPS = {"SHF": C_SHAFT, "H_WJ": C_HOUSING / 4, "H_ATF": C_HOUSING / 4,
            "H_RST": C_HOUSING / 2, "AIR": 10.0}
for nm, c in NET_CAPS.items():
    add_C(NET_NODES[nm], c)

def add_R(n1, n2, conductance):
    _rid[0] += 1
    mapdl.type(3); mapdl.real(_rid[0]); mapdl.r(_rid[0], conductance); mapdl.e(n1, n2)

# 열저항 엣지 (이름1, 이름2, 컨덕턴스 W/K). 10장 시각화에서 재사용.
A_GAP = 2 * math.pi * ((R_ROT_OUT + R_STA_IN) / 2) * STACK
NET_EDGES = [
    ("GAP_S", "GAP_R", TC_AIR * A_GAP / GAP),               # 공극(전도 근사)
    ("H_WJ",  "WJ",    G_WJ),                                # 하우징(WJ부)-냉각수
    ("H_ATF", "ATF",   HTC_ATF * 2 * math.pi * R_STA_OUT * STACK / 4),
    ("H_WJ",  "AMB",   (1 / R_HOUS_AMB) / 4),
    ("H_ATF", "AMB",   (1 / R_HOUS_AMB) / 4),
    ("H_RST", "AMB",   (1 / R_HOUS_AMB) / 2),
    ("H_WJ",  "H_RST", 1 / R_HOUS_AXIAL),
    ("H_ATF", "H_RST", 1 / R_HOUS_AXIAL),
    ("SHF",   "AIR",   1 / R_SHAFT_TH),
    ("AIR",   "H_RST", HTC_AIR * 2 * math.pi * R_STA_OUT * 0.4),
]
for a, b, g in NET_EDGES:
    add_R(NET_NODES[a], NET_NODES[b], g)

NET_FIXED = ("WJ", "ATF", "AMB")   # 고정온도 노드
for nm in NET_FIXED:
    mapdl.d(NET_NODES[nm], "TEMP", T_INIT)
print("thermal network built:", len(NET_EDGES), "R,", len(NET_CAPS), "C")

## 5. SURF152 로 FEM 경계면 ↔ 회로 노드 연결
스테이터 외경을 상부 90°(WJ)/하부 90°(ATF)/나머지(하우징)로 나눠 비대칭 냉각.
코일엔드 냉각은 **재료(구리)+축위치(|z|>STACK/2)** 로 선택(형상무관 근사).

In [ ]:
# ⚠ ESURF 는 '현재 REAL 상수'를 SURF152 에 물린다. 4장에서 회로 real(101+)을
#   쓴 직후라 그대로 두면 COMBIN14 와 real 을 공유해 SOLVE 에서
#   "Real constant N referenced by element types 3 and 2" 에러 발생.
#   -> SURF152 전용 real 1 을 만들고 ESURF 전에 REAL,1 로 고정한다.
mapdl.r(1)   # SURF152 전용 real 상수

def make_surf(node_select_fn, extra_node, htc, name=""):
    mapdl.allsel()
    node_select_fn()
    nsel = mapdl.mesh.n_node
    if nsel == 0:
        print(f"  [warn] {name}: 표면 절점 선택 결과 없음 - 조건 확인"); return
    e0 = int(mapdl.get_value("ELEM", 0, "NUM", "MAXD"))
    mapdl.esln("S", 0)
    mapdl.esel("R", "TYPE", "", 1)             # 솔리드 요소만
    mapdl.nsel("A", "NODE", "", extra_node)
    mapdl.type(2); mapdl.real(1)               # <- real 충돌 방지 핵심
    mapdl.esurf(extra_node)
    e1 = int(mapdl.get_value("ELEM", 0, "NUM", "MAXD"))
    if e1 <= e0:
        print(f"  [warn] {name}: 절점 {nsel}개 선택됐으나 SURF152 미생성"); mapdl.allsel(); return
    mapdl.esel("S", "ELEM", "", e0 + 1, e1)
    mapdl.sfe("ALL", 1, "CONV", "", htc)
    mapdl.allsel()
    print(f"  [surf] {name}: SURF152 {e1 - e0}개 (절점 {nsel})")

def sel_cyl(radius=None, th_ranges=None, z_range=None, sub_bands=None):
    def _fn():
        mapdl.csys(1); mapdl.seltol(TOL)
        # 곡면 사면체의 midside 노드는 현(chord) 위에 놓여 반경이 sagitta
        # (~ESIZE^2/8R)만큼 벗어남 -> 정확반경 선택 시 면이 불완전해져 ESURF 가
        # 요소를 만들지 못한다. 반경 허용오차를 요소크기 기반으로 확대.
        rt = TOL if radius is None else max(TOL, 1.6 * ESIZE**2 / (8 * radius))
        first = True
        for (a, b) in (th_ranges or [(-180.0, 180.0)]):
            key = "S" if first else "A"
            if radius is not None:
                mapdl.nsel(key, "LOC", "X", radius - rt, radius + rt)
                mapdl.nsel("R", "LOC", "Y", a, b)
            else:
                mapdl.nsel(key, "LOC", "Y", a, b)
            if z_range is not None:
                mapdl.nsel("R", "LOC", "Z", z_range[0] - TOL, z_range[1] + TOL)
            if first:
                mapdl.cm("_TMPSEL", "NODE")
            else:
                mapdl.cmsel("A", "_TMPSEL"); mapdl.cm("_TMPSEL", "NODE")
            first = False
        mapdl.cmsel("S", "_TMPSEL")
        if sub_bands:
            for (r1, r2) in sub_bands:
                mapdl.nsel("U", "LOC", "X", r1 + TOL, r2 - TOL)
        mapdl.seltol(0)
    return _fn

# --- (a) 스테이터 외경면 -> 하우징/WJ/ATF (비대칭) ---
make_surf(sel_cyl(radius=R_STA_OUT, th_ranges=TH_WJ),   N_H_WJ,  H_CONTACT, "statorOD-WJ")
make_surf(sel_cyl(radius=R_STA_OUT, th_ranges=TH_REST), N_H_RST, H_CONTACT, "statorOD-REST")
make_surf(sel_cyl(radius=R_STA_OUT, th_ranges=TH_ATF),  N_ATF,   HTC_ATF,   "statorOD-ATF")

# --- (b) 공극 양면 -> 회로노드(저항은 COMBIN14 담당) ---
make_surf(sel_cyl(radius=R_STA_IN),  N_GAP_S, HTC_BIG, "gap-stator")
make_surf(sel_cyl(radius=R_ROT_OUT), N_GAP_R, HTC_BIG, "gap-rotor")

# --- (c) 로터/샤프트 내경면 -> 샤프트 노드 ---
make_surf(sel_cyl(radius=R_SHAFT), N_SHF, HTC_BIG, "shaft-bore")

# --- (d) 축방향 단면(로터측) -> 내부공기 ---
for z in (-STACK / 2, STACK / 2):
    make_surf(sel_cyl(z_range=(z, z), sub_bands=[(R_ROT_OUT, 2 * R_STA_OUT)]),
              N_AIR, HTC_AIR, f"rotor-end z={z:+.3f}")

# --- (e) 코일엔드(구리, 코어 밖) 표면: ATF 섹터 / 나머지 ---
def sel_coilend_simple(th_ranges):
    def _fn():
        mapdl.allsel()
        mapdl.esel("S", "MAT", "", MAT_COIL); mapdl.nsle("S")
        mapdl.csys(1); mapdl.seltol(TOL)
        mapdl.nsel("U", "LOC", "Z", -STACK / 2 + TOL, STACK / 2 - TOL)
        mapdl.cm("_CEALL", "NODE")
        first = True
        for (a, b) in th_ranges:
            if first:
                mapdl.cmsel("S", "_CEALL"); mapdl.nsel("R", "LOC", "Y", a, b)
                mapdl.cm("_CESEL", "NODE"); first = False
            else:
                mapdl.cmsel("S", "_CEALL"); mapdl.nsel("R", "LOC", "Y", a, b)
                mapdl.cmsel("A", "_CESEL"); mapdl.cm("_CESEL", "NODE")
        mapdl.cmsel("S", "_CESEL"); mapdl.seltol(0)
    return _fn

make_surf(sel_coilend_simple(TH_ATF), N_ATF, HTC_ATF)
make_surf(sel_coilend_simple(TH_WJ + TH_REST), N_AIR, HTC_AIR)
print("SURF152 boundaries created")

### 경계(SURF152) 확인 — 표면요소만 표시

In [ ]:
mapdl.esel('S', 'TYPE', '', 2)
mapdl.eplot(vtk=True)
mapdl.allsel()

## 6. 발열 — 재료군별 총손실 → 체적발열률(HGEN). 링/CAD 공통(MAT 기반)

In [ ]:
MAT_LOSS = {
    MAT_CORE_S: LOSS_IRON_STATOR,
    MAT_CORE_R: LOSS_IRON_ROTOR,
    MAT_MAG:    LOSS_MAGNET,
    MAT_COIL:   LOSS_COPPER,
    MAT_SHAFT:  0.0,
}
mapdl.allsel()
for mat, W in MAT_LOSS.items():
    if W <= 0:
        continue
    mapdl.vsel("S", "MAT", "", mat)
    if int(mapdl.get_value("VOLU", 0, "COUNT")) == 0:   # COUNT=선택된 볼륨수 (MAXD 는 선택무관)
        print(f"  [warn] mat {mat}: 볼륨 없음, 손실 스킵"); continue
    mapdl.vsum()
    vol = mapdl.get_value("VOLU", 0, "VOLU")
    q = W / vol if vol > 0 else 0.0
    mapdl.bfv("ALL", "HGEN", q)
    print(f"  mat {mat}: V={vol*1e6:.1f} cm3, {W:.0f} W -> HGEN={q:.3e} W/m3")
mapdl.allsel()

## 7. 과도 해석 (900 s, dt=45 s, 초기 70 degC)  — 메시크기에 따라 수 분

In [ ]:
mapdl.slashsolu()
mapdl.antype(4)              # transient
mapdl.trnopt("FULL")
mapdl.timint(1)
mapdl.ic("ALL", "TEMP", T_INIT)
mapdl.kbc(1)
mapdl.deltim(DT, DT / 3, DT)
mapdl.time(T_END)
mapdl.outres("NSOL", "ALL")
mapdl.solve()
mapdl.finish()
print("solve done")

## 8. 후처리 — 코일 온도 이력 (JAC279 Table 3-1 비교)

In [ ]:
mapdl.post1()
mapdl.csys(0)
r_eval = (R_COIL_IN + R_COIL_OUT) / 2
z_tip = STACK / 2 + COILEND_H / 2

mapdl.esel("S", "MAT", "", MAT_COIL); mapdl.nsle("S")   # 코일 절점만
pts = {
    "Center_WJ":  (0,  r_eval, 0),
    "Center_ATF": (0, -r_eval, 0),
    "Tip_WJ":     (0,  r_eval, z_tip),
    "Tip_ATF":    (0, -r_eval, z_tip),
}
eval_nodes = {k: int(mapdl.queries.node(*xyz)) for k, xyz in pts.items()}
mapdl.allsel()
print("평가 절점:", eval_nodes)

nsets = int(mapdl.get_value("ACTIVE", 0, "SET", "NSET"))
history = []
for i in range(1, nsets + 1):
    mapdl.set(1, i)
    t = mapdl.get_value("ACTIVE", 0, "SET", "TIME")
    row = {"time_s": t}
    for k, n in eval_nodes.items():
        row[k] = mapdl.get_value("NODE", n, "TEMP")
    history.append(row)

out_csv = "jac279_coil_temp.csv"
with open(out_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    w.writeheader(); w.writerows(history)
print(f"온도 이력 저장: {out_csv}")

last = history[-1]
print(f"\n=== t = {last['time_s']:.0f} s 코일 온도 ===")
for k in ("Center_WJ", "Center_ATF", "Tip_WJ", "Tip_ATF"):
    print(f"  {k:12s}: {last[k]:7.1f} degC")

import matplotlib.pyplot as plt

INK, INK2, GRIDC = "#333333", "#666666", "#e5e5e0"
SERIES = {"Center_WJ": "#2a78d6", "Center_ATF": "#1baf7a",
          "Tip_WJ": "#eda100", "Tip_ATF": "#008300"}
ts = [r["time_s"] for r in history]
fig, ax = plt.subplots(figsize=(9, 5.5))
finals = sorted((history[-1][k], k) for k in SERIES)
min_gap = max((finals[-1][0] - finals[0][0] + 4) / max(len(finals) - 1, 1), 1.2)
lab_y, y_prev = {}, None
for v, k in finals:                     # 끝 라벨 세로 겹침 방지 (dodge)
    y = v if y_prev is None else max(v, y_prev + min_gap)
    lab_y[k] = y; y_prev = y
for k, c in SERIES.items():
    ys = [r[k] for r in history]
    ax.plot(ts, ys, color=c, lw=2, label=k)
    ax.annotate(f"{k}  {ys[-1]:.1f}", xy=(ts[-1], ys[-1]),
                xytext=(ts[-1] * 1.01, lab_y[k]), textcoords="data",
                va="center", fontsize=9, color=INK)
ax.set_xlabel("Time, s", color=INK); ax.set_ylabel("Temperature, degC", color=INK)
ax.set_title("Coil temperature history (JAC279-style FEM + thermal network)",
             color=INK, fontsize=12)
ax.grid(True, color=GRIDC, lw=0.8); ax.tick_params(colors=INK2)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRIDC)
ax.legend(frameon=False, fontsize=9, labelcolor=INK)
ax.set_xlim(0, max(ts) * 1.22)
fig.tight_layout()
fig.savefig("jac279_coil_temp.png", dpi=150)
print("그래프 저장: jac279_coil_temp.png")
plt.show()

## 9. FEA 컨투어 시각화 (pyvista)
최종 시각의 온도장을 3개 뷰로 저장/표시:
① 외표면 등각 뷰 ② `x=0` 수직 절단면(상부 WJ ↔ 하부 ATF 비대칭 확인) ③ `z=0` 횡단면.
컬러맵은 지각균일(단조 명도) 순차 램프 `inferno` (온도=크기값).

In [ ]:
import pyvista as pv

mapdl.post1(); mapdl.set("LAST")
t_last = mapdl.get_value("ACTIVE", 0, "SET", "TIME")

import numpy as np

mapdl.esel("S", "TYPE", "", 1); mapdl.nsle("S")     # 솔리드만 (회로/표면요소 제외)
grid = mapdl.mesh.grid
temps = mapdl.post_processing.nodal_temperature().astype(float)
# 미사용/병합 노드의 쓰레기값이 컬러 범위를 오염시킴 -> 물리 범위 밖 마스킹
bad = (temps < T_INIT - 60.0) | (temps > 1000.0)
print(f"masked garbage nodes: {int(bad.sum())} / {temps.size}")
temps[bad] = np.nan
grid.point_data["Temperature (degC)"] = temps
mapdl.allsel()

# 스칼라바 공통 설정. 뷰별로 자체 온도범위(clim)를 사용해 좁은 범위의
# 그라디언트도 읽히게 한다. 표면 뷰는 ambient 보정 조명(형상 인지),
# 절단면은 무조명 평면색(FEA 컨투어 표준). nan_opacity 는 반투명 렌더링을
# 강제하므로 bad>0 일 때만 추가할 것.
sb = dict(title="Temperature (degC)", title_font_size=16, label_font_size=13,
          n_labels=6, fmt="%.1f", color="black")
views = [
    # (파일, 메시, 카메라, 제목, 표면조명 여부)
    ("contour_iso.png",      grid,                   ("view_isometric",),
     f"surface T @ t={t_last:.0f}s", True),
    ("contour_slice_x0.png", grid.slice(normal="x"),
     ("view_vector", (1, 0, 0), dict(viewup=(0, 1, 0))),   # y(WJ쪽)=화면 위
     f"x=0 slice (top: WJ / bottom: ATF) @ t={t_last:.0f}s", False),
    ("contour_slice_z0.png", grid.slice(normal="z"), ("view_xy",),
     f"z=0 cross-section @ t={t_last:.0f}s", False),
]
for fname, mesh, view, title, lit in views:
    tv = mesh.point_data["Temperature (degC)"]
    kw = dict(cmap="inferno", scalar_bar_args=sb,
              clim=[float(np.nanmin(tv)), float(np.nanmax(tv))])
    if lit:
        kw.update(smooth_shading=True, ambient=0.6, diffuse=0.4, specular=0.0)
    else:
        kw.update(lighting=False)
    p = pv.Plotter(window_size=(1280, 960))
    p.set_background("white")
    p.add_mesh(mesh, **kw)
    p.add_text(title, font_size=12, color="black")
    getattr(p, view[0])(*view[1:2], **(view[2] if len(view) > 2 else {}))
    p.screenshot(fname)         # PNG 저장
    p.close()
    print("저장:", fname)

# 인터랙티브 뷰 (Jupyter 내)
p = pv.Plotter(window_size=(900, 700))
p.set_background("white")
sx = grid.slice(normal="x")
tv = sx.point_data["Temperature (degC)"]
p.add_mesh(sx, cmap="inferno", lighting=False, scalar_bar_args=sb,
           clim=[float(np.nanmin(tv)), float(np.nanmax(tv))])
p.view_vector((1, 0, 0), viewup=(0, 1, 0))
p.show()

## 10. 열등가회로 시각화
4장의 `NET_NODES`/`NET_EDGES`/`NET_CAPS` 와 해석결과 온도로 회로도를 그린다.
- **노드 색 = 온도** (FEA 컨투어와 동일한 `inferno` 램프, 공용 컬러바)
- 사각형 = 고정온도(D), 원 = 부동 노드, 실선 = COMBIN14(라벨: 컨덕턴스), 점선 = SURF152 결합
- 좌측 박스 = FEM 영역(재료별 평균/최대 온도)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch

mapdl.post1(); mapdl.set("LAST")

# 회로 노드 온도
node_T = {nm: mapdl.get_value("NODE", n, "TEMP") for nm, n in NET_NODES.items()}
print("회로 노드 온도:", {k: round(v, 1) for k, v in node_T.items()})

# FEM 영역(재료별) 평균/최대 온도
reg_T = {}
for nm, mat in (("StatorCore", MAT_CORE_S), ("RotorCore", MAT_CORE_R),
                ("Magnet", MAT_MAG), ("Coil", MAT_COIL), ("Shaft", MAT_SHAFT)):
    mapdl.allsel(); mapdl.esel("S", "MAT", "", mat); mapdl.nsle("S")
    arr = mapdl.post_processing.nodal_temperature()
    arr = arr[~np.isnan(arr)]
    if arr.size:
        reg_T[nm] = (float(arr.mean()), float(arr.max()))
mapdl.allsel()
print("FEM 영역 온도(avg,max):", {k: (round(a,1), round(b,1)) for k, (a,b) in reg_T.items()})

INK, INK2 = "#333333", "#666666"
allT = list(node_T.values()) + [a for (a, _b) in reg_T.values()]
vmin, vmax_ = min(allT), max(max(allT), min(allT) + 1)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax_)
cmap = cm.get_cmap("inferno")
tcolor = lambda t: cmap(norm(t))
def txtcol(t):
    r, g, b, _ = tcolor(t)
    return "#ffffff" if (0.299*r + 0.587*g + 0.114*b) < 0.55 else "#1a1a19"

pos = {"AMB": (0.5, 4.1),
       "H_WJ": (-1.6, 3.0), "H_RST": (0.5, 3.0), "H_ATF": (2.6, 3.0),
       "WJ": (-3.2, 3.0), "ATF": (4.2, 3.0), "AIR": (2.6, 1.55),
       "GAP_S": (-0.2, 0.0), "GAP_R": (1.2, 0.0), "SHF": (2.6, -1.4)}

fig, ax = plt.subplots(figsize=(12.5, 9))
ax.set_xlim(-4.4, 6.3); ax.set_ylim(-3.6, 4.9); ax.axis("off")
ax.set_title(f"Thermal equivalent circuit - node temperatures @ t={mapdl.get_value('ACTIVE',0,'SET','TIME'):.0f}s",
             color=INK, fontsize=13)

# COMBIN14 엣지 (실선 + 컨덕턴스 라벨)
for a, b, g in NET_EDGES:
    xa, ya = pos[a]; xb, yb = pos[b]
    rad = 0.18 if {a, b} == {"AIR", "H_RST"} else 0.0
    ax.annotate("", xy=(xb, yb), xytext=(xa, ya),
                arrowprops=dict(arrowstyle="-", color="#b9b8ad", lw=1.6,
                                connectionstyle=f"arc3,rad={rad}"))
    xm = (xa + xb) / 2 + rad * (yb - ya) * 0.9
    ym = (ya + yb) / 2 - rad * (xb - xa) * 0.9
    ax.text(xm, ym, f"{g:.2f} W/K", fontsize=7.5, color=INK2, ha="center",
            va="center", bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.6))

# FEM 영역 박스 (평균온도 색)
fem_boxes = [("FEM Stator core", "StatorCore", -3.9, 1.0, 3.0, 1.1),
             ("FEM Coil (+ends)", "Coil", -3.9, -0.6, 3.0, 1.1),
             ("FEM Rotor core", "RotorCore", -3.9, -2.2, 1.6, 1.1),
             ("FEM Magnet", "Magnet", -2.1, -2.2, 1.2, 1.1),
             ("FEM Shaft", "Shaft", -3.9, -3.5, 3.0, 0.9)]
for lab, key, x, y, w, h in fem_boxes:
    if key not in reg_T:
        continue                     # 링 경로엔 샤프트 볼륨 없음 등
    Tavg, Tmax = reg_T[key]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                fc=tcolor(Tavg), ec="#8a8878", lw=1.2))
    ax.text(x + w/2, y + h/2, f"{lab}\navg {Tavg:.1f} / max {Tmax:.1f}",
            ha="center", va="center", fontsize=8.5, color=txtcol(Tavg))

# SURF152 결합 (점선)
surf_links = [((-0.9, 2.1), "H_WJ", "stator OD 90 (h=3000)"),
              ((-0.9, 2.1), "H_RST", "stator OD 180"),
              ((-0.9, 2.1), "ATF", "stator OD 90 (h=300)"),
              ((-0.9, 1.55), "GAP_S", "stator bore"),
              ((-0.9, -1.65), "GAP_R", "rotor OD"),
              ((-0.9, -3.05), "SHF", "shaft bore"),
              ((-0.9, -0.05), "ATF", "coil-end ATF 90"),
              ((-0.9, -0.05), "AIR", "coil-end rest")]
for (xa, ya), b, lab in surf_links:
    xb, yb = pos[b]
    ax.annotate("", xy=(xb, yb), xytext=(xa, ya),
                arrowprops=dict(arrowstyle="-", color="#8a8878", lw=1.0,
                                linestyle=(0, (4, 3))))
    ax.text((xa+xb)/2, (ya+yb)/2 + 0.09, lab, fontsize=6.8, color=INK2,
            ha="center", bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.4))

# 회로 노드 (사각=고정온도, 원=부동)
for nm, (x, y) in pos.items():
    t = node_T[nm]
    if nm in NET_FIXED:
        ax.add_patch(plt.Rectangle((x-0.42, y-0.3), 0.84, 0.6,
                                   fc=tcolor(t), ec="#1a1a19", lw=1.6))
    else:
        ax.add_patch(plt.Circle((x, y), 0.4, fc=tcolor(t), ec="#8a8878", lw=1.4))
    ax.text(x, y+0.07, nm, ha="center", va="center", fontsize=9,
            color=txtcol(t), fontweight="bold")
    ax.text(x, y-0.17, f"{t:.1f}", ha="center", va="center", fontsize=8,
            color=txtcol(t))
    if nm in NET_CAPS:
        ax.text(x, y-0.56, f"C={NET_CAPS[nm]:.0f} J/K", ha="center",
                fontsize=7, color=INK2)

ax.text(-4.2, 4.55, "square = fixed T   circle = floating node   "
        "dashed = SURF152 coupling   solid = COMBIN14 (label: G)",
        fontsize=8.5, color=INK2)
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.01)
cb.set_label("Temperature (degC)", color=INK)
cb.ax.tick_params(colors=INK2)
fig.tight_layout()
fig.savefig("thermal_circuit.png", dpi=150)
print("저장: thermal_circuit.png")
plt.show()

## 세션 종료
결과 검토가 끝난 뒤에만 실행. 커널이 살아있는 동안 MAPDL 라이선스가 유지된다.

In [ ]:
mapdl.exit()